# EDA Datasets Inventory

Inventory of available datasets and storage footprint.

Steps:
- Scan key data directories.
- Summarize file counts and sizes.
- Review training data audit summary.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from collections import Counter
from pathlib import Path

summary = {
    'directories': {},
    'largest_files': [],
    'training_data': {},
}


def format_bytes(num: int) -> str:
    step = 1024.0
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    size = float(num)
    for unit in units:
        if size < step:
            return f'{size:,.1f} {unit}'
        size /= step
    return f'{size:,.1f} PB'


def scan_dir(path: Path, max_files: int = 200000):
    counts = 0
    total_size = 0
    ext_counts = Counter()
    largest = []
    truncated = False
    for file in path.rglob('*'):
        if not file.is_file():
            continue
        counts += 1
        try:
            size = file.stat().st_size
        except OSError:
            size = 0
        total_size += size
        ext_counts[file.suffix.lower() or 'no_ext'] += 1
        largest.append((size, file))
        largest = sorted(largest, key=lambda item: item[0], reverse=True)[:5]
        if counts >= max_files:
            truncated = True
            break
    return {
        'files': counts,
        'size_bytes': total_size,
        'size_human': format_bytes(total_size),
        'top_extensions': dict(ext_counts.most_common(6)),
        'largest_files': [(size, str(path.relative_to(REPO_ROOT))) for size, path in largest],
        'truncated': truncated,
    }


roots = {
    'data/raw': REPO_ROOT / 'data' / 'raw',
    'data/processed': REPO_ROOT / 'data' / 'processed',
    'data/docs': REPO_ROOT / 'data' / 'docs',
    'data/dsa_docs': REPO_ROOT / 'data' / 'dsa_docs',
    'data/samples': REPO_ROOT / 'data' / 'samples',
    'data/Celeb_V2': REPO_ROOT / 'data' / 'Celeb_V2',
}

largest_overall = []

for name, root in roots.items():
    print('Scanning:', name)
    if not root.exists():
        print('Missing:', root)
        continue
    stats = scan_dir(root)
    summary['directories'][name] = stats
    print('Files:', stats['files'], 'Size:', stats['size_human'])
    print('Top extensions:', stats['top_extensions'])
    if stats['truncated']:
        print('Scan truncated for', name)
    for size, path_str in stats['largest_files']:
        largest_overall.append((size, path_str))

largest_overall = sorted(largest_overall, key=lambda item: item[0], reverse=True)[:8]
summary['largest_files'] = [
    {'path': path_str, 'size_bytes': size, 'size_human': format_bytes(size)}
    for size, path_str in largest_overall
]

print('Largest files:')
for item in summary['largest_files']:
    print(' -', item['path'], item['size_human'])


In [ ]:
# Summarize training data audit if available.
training_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if training_path.exists():
    data = json.loads(training_path.read_text(encoding='utf-8'))
    required = data.get('required', [])
    optional = data.get('optional', [])
    missing_required = [item for item in required if item.get('status') != 'ok']
    missing_optional = [item for item in optional if item.get('status') != 'ok']
    summary['training_data'] = {
        'required': len(required),
        'optional': len(optional),
        'missing_required': len(missing_required),
        'missing_optional': len(missing_optional),
    }
    print('Required datasets:', len(required))
    print('Optional datasets:', len(optional))
    print('Missing required:', len(missing_required))
    print('Missing optional:', len(missing_optional))
    for item in missing_required[:10]:
        print(' -', item.get('name'), item.get('status'))
else:
    print('Missing TRAINING_DATA.json')


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_datasets_inventory_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize data-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'data' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No data entries found in TRAINING_DATA.json')
    else:
        print('data datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
